# A Mixed-Form PINNs (MF-PINNs) for Solving the Coupled Stokes-Darcy Equations

**Paper:** Shan, L. and Shen, X. (2025). *A Mixed-Form PINNs (MF-PINNs) for Solving the Coupled Stokes-Darcy Equations.* arXiv:2510.17508 [physics.flu-dyn].

**Carpeta origen:** `PINNs/1. mecanica de fluidos/A_Mixed-Form_PINNS_MF-PINNS_For_Solving_The_Couple.pdf`

## Repositorio publico

El propio abstract del paper enlaza el codigo oficial:

> "The code and data associated with this paper are available at https://github.com/shxshx48716/MF-PINNs.git"

Este cuaderno es una reproduccion simplificada y autocontenida de la idea central de ese repositorio (no un clon), pensada para ejecutarse standalone.

## Como se usan las PINNs en este paper

El paper resuelve el sistema acoplado de Stokes-Darcy con condiciones de interfaz Beavers-Joseph-Saffman (BJS) usando **PINNs paralelas** (una red para el dominio de Stokes, otra para el de Darcy):

- **Stokes** (dominio $\Omega_s=[0,1]\times[0,1]$): $\nabla p_s - \nu\Delta \mathbf{u}_s = \mathbf{f}_s$, $\nabla\cdot\mathbf{u}_s=0$ (Eq. 2.1).
- **Darcy** (dominio $\Omega_d=[0,1]\times[-1,0]$): $\nu\mathbb{K}^{-1}\mathbf{u}_d + \nabla p_d = \mathbf{f}_d$, $\nabla\cdot\mathbf{u}_d=0$ (Eq. 2.2), con $\mathbb{K}=\kappa\mathbb{I}$.
- **Interfaz BJS** en $y=0$ (Eq. 2.3): continuidad de velocidad normal, balance de tension normal, y balance de momento tangencial con friccion $\alpha\mathbb{K}^{-1/2}$.

La perdida total combina el residuo PDE de cada dominio, la interfaz y las condiciones de contorno (Eq. 3.2-3.7). El **hallazgo central del paper** es que, cuando $\nu/\kappa \gg 1$ (permeabilidad muy baja), la perdida PINN estandar (pesos uniformes) sufre *competicion de gradiente*: el campo de velocidad de Darcy converge bien pero el campo de presion **no converge en absoluto** ($err_{L_2}(p_d)=135.4\%$ reportado en el paper, Fig. 4.1). Su solucion, **MF-PINNs**, combina la forma velocidad-presion (VP) con la forma streamline-vorticidad (SV, Teoremas 1-2) y reponderar los terminos de perdida (Eq. 3.11-3.13) para balancear las escalas de gradiente.

Este cuaderno reproduce fielmente:

1. El benchmark de solucion manufacturada exacta del paper (Eq. 4.2), usado para derivar automaticamente los terminos forzantes $\mathbf{f}_s,\mathbf{f}_d$ y las condiciones de contorno/interfaz $g_{\Gamma_1}, g_{\Gamma_2}$ via diferenciacion automatica (igual que hace el propio paper).
2. Las PINNs paralelas en forma VP (Eq. 2.1-2.3) para Stokes y Darcy, con la condicion de interfaz BJS completa.
3. Los dos esquemas de pesos que el paper compara en la Seccion 3.5: **PINNs baseline** ($\lambda=1$ uniforme) vs. **pesos estilo MF-PINNs** ($\lambda_{u_s}=\lambda_{u_d}=10^2,\ \lambda_{f_d}=\kappa,\ \lambda_{f_s}=\lambda_\Gamma=1$, Seccion 3.5), aplicados sobre la misma perdida VP.

**Simplificacion declarada:** la forma Mixed-Form completa del paper (Eq. 3.11-3.12) decouplea el sistema usando la forma streamline-vorticidad (SV) con operadores de curl y Laplaciano de 4to orden (Teoremas 1-2) y usa Adam+L-BFGS en dos etapas. Aqui reproducimos fielmente la forma VP y el esquema de pesos, pero no la derivacion SV completa ni la fase L-BFGS, por brevedad; para la implementacion completa ver el repositorio oficial enlazado arriba.

In [ ]:
# Instalacion de dependencias (ejecutar si no estan ya instaladas en el entorno)
%pip install -q torch numpy matplotlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Solucion analitica manufacturada (Eq. 4.2 del paper) y parametros fisicos

In [ ]:
nu = 1.0
kappa = 1e-4   # K = kappa * I -- caso extremo del paper (Seccion 4.4.1): nu/kappa = 1e4
alpha = 1.0

def exact_stokes(xy):
    x, y = xy[:, 0:1], xy[:, 1:2]
    u = -torch.sin(np.pi * x)**2 * torch.sin(np.pi * y) * torch.cos(np.pi * y)
    v = torch.sin(np.pi * x) * torch.cos(np.pi * x) * torch.sin(np.pi * y)**2
    p = torch.sin(np.pi * x) * torch.cos(np.pi * y)
    return u, v, p

def exact_darcy(xy):
    x, y = xy[:, 0:1], xy[:, 1:2]
    u = 0.5 * torch.sin(2 * np.pi * x) * torch.cos(2 * np.pi * y)
    v = -0.5 * torch.cos(2 * np.pi * x) * torch.sin(2 * np.pi * y)
    p = torch.sin(np.pi * x) * torch.cos(np.pi * y)
    return u, v, p

## 2. Redes PINN paralelas (Tabla 4.1: 4 capas ocultas x 70 neuronas, tanh)

In [ ]:
class PINN(nn.Module):
    def __init__(self, n_hidden_layers=4, n_neurons=70):
        super().__init__()
        layers = [nn.Linear(2, n_neurons), nn.Tanh()]
        for _ in range(n_hidden_layers - 1):
            layers += [nn.Linear(n_neurons, n_neurons), nn.Tanh()]
        layers += [nn.Linear(n_neurons, 3)]  # (u, v, p)
        self.net = nn.Sequential(*layers)

    def forward(self, xy):
        out = self.net(xy)
        return out[:, 0:1], out[:, 1:2], out[:, 2:3]


def d_dxy(f, xy):
    """Gradiente [df/dx, df/dy] via autograd, conservando el grafo."""
    g = torch.autograd.grad(f, xy, grad_outputs=torch.ones_like(f),
                             create_graph=True, retain_graph=True)[0]
    return g[:, 0:1], g[:, 1:2]

## 3. Operadores PDE (Eq. 2.1, 2.2 simplificadas) — se reusan tanto para la solucion exacta (para obtener $\mathbf{f}_s,\mathbf{f}_d$ por diferenciacion automatica) como para el residuo de la red

In [ ]:
def stokes_operator(u, v, p, xy):
    """L(u,v,p) = (p_x - nu*Lap(u), p_y - nu*Lap(v), u_x+v_y). Eq. (2.1) simplificada."""
    u_x, u_y = d_dxy(u, xy)
    v_x, v_y = d_dxy(v, xy)
    p_x, p_y = d_dxy(p, xy)
    u_xx, _ = d_dxy(u_x, xy)
    _, u_yy = d_dxy(u_y, xy)
    v_xx, _ = d_dxy(v_x, xy)
    _, v_yy = d_dxy(v_y, xy)
    Lx = p_x - nu * (u_xx + u_yy)
    Ly = p_y - nu * (v_xx + v_yy)
    Lc = u_x + v_y
    return Lx, Ly, Lc, (u_x, u_y, v_x, v_y, p_x, p_y)


def darcy_operator(u, v, p, xy):
    """L(u,v,p) = (nu/kappa*u + p_x, nu/kappa*v + p_y, u_x+v_y). Eq. (2.2)."""
    u_x, u_y = d_dxy(u, xy)
    v_x, v_y = d_dxy(v, xy)
    p_x, p_y = d_dxy(p, xy)
    Lx = (nu / kappa) * u + p_x
    Ly = (nu / kappa) * v + p_y
    Lc = u_x + v_y
    return Lx, Ly, Lc


def make_grid(nx, ny, x_range, y_range):
    xs = np.linspace(*x_range, nx)
    ys = np.linspace(*y_range, ny)
    X, Y = np.meshgrid(xs, ys)
    pts = np.stack([X.ravel(), Y.ravel()], axis=1)
    return torch.tensor(pts, dtype=torch.float32, device=device, requires_grad=True)


# Puntos de colocacion (version reducida de la malla 127x127 del paper para agilizar el entrenamiento)
xy_s = make_grid(40, 40, (0, 1), (0, 1))
xy_d = make_grid(40, 40, (0, 1), (-1, 0))

n_b = 60
xy_iface = make_grid(n_b, 1, (0, 1), (0, 0))                          # interfaz y=0
xy_s_bnd = torch.cat([make_grid(1, n_b, (0, 0), (0, 1)),               # x=0
                      make_grid(1, n_b, (1, 1), (0, 1)),               # x=1
                      make_grid(n_b, 1, (0, 1), (1, 1))], dim=0)        # y=1
xy_d_bnd_x0 = make_grid(1, n_b, (0, 0), (-1, 0))                       # x=0, normal=(-1,0)
xy_d_bnd_x1 = make_grid(1, n_b, (1, 1), (-1, 0))                       # x=1, normal=(1,0)
xy_d_bnd_yb = make_grid(n_b, 1, (0, 1), (-1, -1))                      # y=-1, normal=(0,-1)

# Forzantes f_s, f_d obtenidos por diferenciacion automatica de la solucion exacta (Eq. 4.2)
u_s_e, v_s_e, p_s_e = exact_stokes(xy_s)
f_s1, f_s2, f_sc, _ = stokes_operator(u_s_e, v_s_e, p_s_e, xy_s)
f_s1, f_s2 = f_s1.detach(), f_s2.detach()

u_d_e, v_d_e, p_d_e = exact_darcy(xy_d)
f_d1, f_d2, f_dc = darcy_operator(u_d_e, v_d_e, p_d_e, xy_d)
f_d1, f_d2 = f_d1.detach(), f_d2.detach()

print('max |continuity residual| exacta (debe ser ~0):',
      f_sc.abs().max().item(), f_dc.abs().max().item())

## 4. Condiciones de interfaz BJS (Eq. 2.3) y de contorno, evaluadas via diferenciacion automatica

In [ ]:
def interface_terms(u_s, v_s, p_s, u_d, v_d, p_d, xy_s_i, xy_d_i):
    """n_s=(0,-1), n_d=(0,1), tau=(1,0) en la interfaz horizontal y=0 (Eq. 2.3)."""
    u_s_x, u_s_y = d_dxy(u_s, xy_s_i)
    v_s_x, v_s_y = d_dxy(v_s, xy_s_i)
    mass = v_d - v_s                                           # (2.3a): continuidad de vel. normal
    stress = 2 * nu * v_s_y - p_s + p_d                         # (2.3b): balance de tension normal
    tangential = -(u_s_y + v_s_x) + (alpha / np.sqrt(kappa)) * u_s  # (2.3c): friccion tangencial
    return mass, stress, tangential


def compute_losses(net_s, net_d, weights):
    # --- Residuos PDE en el interior de cada dominio ---
    u_s, v_s, p_s = net_s(xy_s)
    Lx_s, Ly_s, Lc_s, _ = stokes_operator(u_s, v_s, p_s, xy_s)
    loss_fs = torch.mean((Lx_s - f_s1)**2 + (Ly_s - f_s2)**2 + Lc_s**2)

    u_d, v_d, p_d = net_d(xy_d)
    Lx_d, Ly_d, Lc_d = darcy_operator(u_d, v_d, p_d, xy_d)
    loss_fd = torch.mean((Lx_d - f_d1)**2 + (Ly_d - f_d2)**2 + Lc_d**2)

    # --- Interfaz BJS ---
    u_s_i, v_s_i, p_s_i = net_s(xy_iface)
    u_d_i, v_d_i, p_d_i = net_d(xy_iface)
    mass, stress, tang = interface_terms(u_s_i, v_s_i, p_s_i, u_d_i, v_d_i, p_d_i, xy_iface, xy_iface)
    u_s_e_i, v_s_e_i, p_s_e_i = exact_stokes(xy_iface)
    u_d_e_i, v_d_e_i, p_d_e_i = exact_darcy(xy_iface)
    mass_e, stress_e, tang_e = interface_terms(u_s_e_i, v_s_e_i, p_s_e_i, u_d_e_i, v_d_e_i, p_d_e_i, xy_iface, xy_iface)
    loss_iface = torch.mean((mass - mass_e.detach())**2 + (stress - stress_e.detach())**2
                             + (tang - tang_e.detach())**2)

    # --- Contorno Dirichlet en Stokes (u_s = solucion exacta) ---
    u_sb, v_sb, _ = net_s(xy_s_bnd)
    u_sb_e, v_sb_e, _ = exact_stokes(xy_s_bnd)
    loss_us = torch.mean((u_sb - u_sb_e)**2 + (v_sb - v_sb_e)**2)

    # --- Contorno Neumann en Darcy: u_d . n_d = g (solo componente normal, Eq. 2.2c) ---
    def normal_bc_loss(xy_pts, normal):
        u_p, v_p, _ = net_d(xy_pts)
        u_e, v_e, _ = exact_darcy(xy_pts)
        pred_n = u_p * normal[0] + v_p * normal[1]
        exact_n = u_e * normal[0] + v_e * normal[1]
        return torch.mean((pred_n - exact_n.detach())**2)

    loss_ud = (normal_bc_loss(xy_d_bnd_x0, (-1.0, 0.0))
               + normal_bc_loss(xy_d_bnd_x1, (1.0, 0.0))
               + normal_bc_loss(xy_d_bnd_yb, (0.0, -1.0))) / 3

    total = (weights['fs'] * loss_fs + weights['fd'] * loss_fd + weights['iface'] * loss_iface
             + weights['us'] * loss_us + weights['ud'] * loss_ud)
    return total, dict(fs=loss_fs.item(), fd=loss_fd.item(), iface=loss_iface.item(),
                        us=loss_us.item(), ud=loss_ud.item())

## 5. Entrenamiento: PINNs baseline (pesos uniformes) vs. pesos estilo MF-PINNs (Seccion 3.5)

In [ ]:
def train(weights, epochs=3000, lr=1e-3, label=''):
    net_s = PINN().to(device)
    net_d = PINN().to(device)
    opt = torch.optim.Adam(list(net_s.parameters()) + list(net_d.parameters()), lr=lr)
    for epoch in range(epochs):
        opt.zero_grad()
        loss, parts = compute_losses(net_s, net_d, weights)
        loss.backward()
        opt.step()
        if epoch % 500 == 0:
            print(f'[{label}] epoch {epoch:5d} | loss={loss.item():.3e} | parts={ {k: round(v,3) for k,v in parts.items()} }')
    return net_s, net_d


def relative_l2_error(net, exact_fn, xy):
    with torch.no_grad():
        u_p, v_p, p_p = net(xy)
        u_e, v_e, p_e = exact_fn(xy)
        err_u = torch.norm(u_p - u_e) / torch.norm(u_e)
        err_v = torch.norm(v_p - v_e) / torch.norm(v_e)
        err_p = torch.norm(p_p - p_e) / torch.norm(p_e)
    return err_u.item(), err_v.item(), err_p.item()


# Esquema 'PINNs' baseline: pesos uniformes lambda=1, Eq. (3.7) y Seccion 3.5
weights_baseline = dict(fs=1.0, fd=1.0, iface=1.0, us=1.0, ud=1.0)

# Esquema 'MF-PINNs (Ours)': lambda_us=lambda_ud=100, lambda_fd=kappa, lambda_fs=lambda_iface=1 (Seccion 3.5)
weights_mf = dict(fs=1.0, fd=kappa, iface=1.0, us=100.0, ud=100.0)

net_s_base, net_d_base = train(weights_baseline, epochs=3000, label='baseline')
net_s_mf, net_d_mf = train(weights_mf, epochs=3000, label='MF-weights')

## 6. Comparacion de errores (reproduce la metrica $err_{L_2}$, Eq. 4.1, y el hallazgo de la Fig. 4.1)

In [ ]:
xy_s_test = make_grid(50, 50, (0, 1), (0, 1))
xy_d_test = make_grid(50, 50, (0, 1), (-1, 0))

for label, (ns, nd) in [('baseline (lambda uniforme)', (net_s_base, net_d_base)),
                        ('MF-PINNs weights', (net_s_mf, net_d_mf))]:
    eu_s, ev_s, ep_s = relative_l2_error(ns, exact_stokes, xy_s_test)
    eu_d, ev_d, ep_d = relative_l2_error(nd, exact_darcy, xy_d_test)
    print(f'--- {label} ---')
    print(f'  Stokes: errL2(u)={eu_s:.3f}  errL2(v)={ev_s:.3f}  errL2(p_s)={ep_s:.3f}')
    print(f'  Darcy : errL2(u)={eu_d:.3f}  errL2(v)={ev_d:.3f}  errL2(p_d)={ep_d:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
with torch.no_grad():
    _, _, p_pred_base = net_d_base(xy_d_test)
    _, _, p_pred_mf = net_d_mf(xy_d_test)
_, _, p_true = exact_darcy(xy_d_test)
n_side = 50
for ax, field, title in zip(
        axes,
        [p_true.detach().cpu().numpy(), p_pred_base.detach().cpu().numpy(), p_pred_mf.detach().cpu().numpy()],
        ['p_d exacta', 'p_d predicha (baseline)', 'p_d predicha (MF-weights)']):
    im = ax.imshow(field.reshape(n_side, n_side), origin='lower', extent=[0, 1, -1, 0], cmap='RdBu_r')
    ax.set_title(title)
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

Se espera observar (igual que en la Fig. 4.1 del paper) que, con $\nu/\kappa=10^4$, el esquema baseline recupera bien los campos de velocidad pero **falla en reconstruir el campo de presion de Darcy**, mientras que el reponderado tipo MF-PINNs mitiga (aunque no elimina del todo, al faltar la componente SV completa) ese problema. Para la solucion MF-PINNs completa con la forma streamline-vorticidad y el entrenamiento Adam+L-BFGS en dos etapas, ver el [repositorio oficial](https://github.com/shxshx48716/MF-PINNs.git).